## requirement def

This notebook introduces `requirement def`; after running it you can declare a formal requirement with a subject and a constraint expression.

Chapter 1 established the structure of the toaster: `Toaster` composes `HeatingSystem` and `ControlSystem`, which specialize `ToastingSystem`. This notebook adds the first requirement: the toaster must complete a cycle in at most 180 seconds. A requirement in SysML v2 has a subject (the part being required), a constraint body, and optionally a documentation comment.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }

    part def Heater {
        attribute power : Real default = 800.0;
    }

    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;

    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }

    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: a constraint that references a non-existent attribute
# raises "unresolved member" at the point of use.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Toaster { attribute cycleTime : Real default = 120.0; }
    requirement def BadReq {
        subject t : Toaster;
        require constraint { t.nonExistentAttr <= 180.0 }
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
req = model.find("ToasterDemo::TimelyToast")
assert req is not None
print(f"requirement kind: {req.kind}")
print(f"requirement id  : {req.id}")

for e in model.query():
    d = e.as_dict()
    if d["@type"] == "RequirementDefinition":
        print(f"RequirementDefinition: {d['qualifiedName']}")
conn.close()

`requirement def TimelyToast { subject toaster : Toaster; require constraint { toaster.cycleTime <= 180.0 } }` is the A-F declaration; OpenSysML parses the constraint and registers the requirement (O-S); `model.find()` returns the symbol and `model.query()` lists it as a RequirementDefinition (E).

Try the chapter exercise in `exercises/ch02/exercise.ipynb`: declare a `TemperatureReq` that requires `brewTemp <= 96.0` and confirm it loads.